# Step 6 - Perturbation engine

How the graph is damaged, and a check that every kind of damage does what it should on the real graph.
Logic is in `src/perturb.py`.

| kind | what it simulates |
|---|---|
| `remove` | missing citations - a fraction of edges is deleted |
| `add_random` | noisy citations, as written in Phase A - new edges between any two papers, in any direction |
| `add_dag` | noisy citations that still respect time - new edges only go from newer to older papers, so no cycle appears (D10) |

In [1]:
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('Final-Project'):
    !git clone https://github.com/ShaharBar99/Final-Project.git
    %cd Final-Project
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

## 1. Why not use the PubMed ID as time for `add_dag`?

We checked whether real citations go from higher ID (newer) to lower ID (older):

In [2]:
import pandas as pd
from src import config, data, perturb

edges = pd.read_csv(config.PROCESSED_DIR / 'edges.csv', dtype=str)
newer_to_older = (edges['source'].astype(int) > edges['target'].astype(int)).mean()
print(f'{newer_to_older:.1%} of the real citations go from a higher to a lower PubMed ID')

89.6% of the real citations go from a higher to a lower PubMed ID


About 10 % do not, so the ID is only a rough proxy. Instead `add_dag` uses a **topological order** of the cleaned graph: an order in which every existing citation already points forward. It exists because the cleaned graph has no cycles (step 1), and adding edges only in that direction can never create one.

## 2. One run of every kind and level - checks

In [3]:
import networkx as nx

g, nodes = data.load_clean_graph()
order = perturb.time_order(g)
rows = []
for kind in perturb.KINDS:
    for level in [0.05, 0.10, 0.20]:
        d = perturb.perturb(g, kind, level, perturb.run_rng(config.SEED, kind, level, 0), order)
        rows.append({
            'kind': kind, 'level': level,
            'edges': d.number_of_edges(),
            'edge change': d.number_of_edges() - g.number_of_edges(),
            'same nodes': set(d.nodes) == set(g.nodes),
            'kept old edges': len(set(d.edges) & set(g.edges)),
            'no cycles': nx.is_directed_acyclic_graph(d),
            'largest cycle group': max(len(c) for c in nx.strongly_connected_components(d)),
            'papers with no edges': sum(1 for n in d if d.degree(n) == 0),
        })
checks = pd.DataFrame(rows)
checks

,kind,level,edges,edge change,same nodes,kept old edges,no cycles,largest cycle group,papers with no edges
0,remove,0.05,42108,-2216,True,42108,True,1,469
1,remove,0.10,39892,-4432,True,39892,True,1,943
2,remove,0.20,35459,-8865,True,35459,True,1,1981
3,add_random,0.05,46540,2216,True,44324,False,328,0
4,add_random,0.10,48756,4432,True,44324,False,901,0
5,add_random,0.20,53189,8865,True,44324,False,2498,0
6,add_dag,0.05,46540,2216,True,44324,True,1,0
7,add_dag,0.10,48756,4432,True,44324,True,1,0
8,add_dag,0.20,53189,8865,True,44324,True,1,0


What to expect, and what we see:
- the node set never changes;
- `remove` deletes exactly 5 / 10 / 20 % of the 44,324 edges and adds none; some papers lose their only citation and become isolated (they stay in the graph with zero degree);
- both `add_*` kinds keep every old edge and add exactly 5 / 10 / 20 % new ones;
- `add_dag` never creates a cycle; `add_random` does, and at 20 % a group of thousands of papers all reach each other - something that cannot happen in a real citation network.

In [4]:
assert checks['same nodes'].all()
assert (checks.query('kind == "add_dag"')['no cycles']).all()
assert (checks.query('kind != "remove"')['kept old edges'] == g.number_of_edges()).all()
print('all checks passed')

all checks passed


## 3. Reproducibility

Every run gets its own random generator built from (seed, kind, level, run), so it can be rebuilt alone.

In [5]:
a = perturb.perturb(g, 'add_dag', 0.1, perturb.run_rng(config.SEED, 'add_dag', 0.1, 3), order)
b = perturb.perturb(g, 'add_dag', 0.1, perturb.run_rng(config.SEED, 'add_dag', 0.1, 3), order)
print('same run twice gives the same graph:', set(a.edges) == set(b.edges))

same run twice gives the same graph: True
